# USER BEHAVIOR PREDICTION — DATAFLOW 2026
## LightGBM 

## 1. Import & Config

Phần này là phần import các thư viện cần thiết, khai báo các siêu tham số và các đường dẫn của dữ liệu.

In [5]:
import os, random, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.sparse import csr_matrix

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score, classification_report,
)
# ✅ THÊM: joblib để parallel feature extraction
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
print(f"✅ LightGBM version: {lgb.__version__}")

✅ LightGBM version: 4.6.0


In [6]:
CFG = {
    # ── Đường dẫn ──────────────────────────────────────────────────
    "x_train":     "/kaggle/input/datasets/thinhdeeptry123/dataset3/X_train.csv",
    "y_train":     "/kaggle/input/datasets/thinhdeeptry123/dataset3/Y_train.csv",
    "x_val":       "/kaggle/input/datasets/thinhdeeptry123/dataset3/X_val.csv",
    "y_val":       "/kaggle/input/datasets/thinhdeeptry123/dataset3/Y_val.csv",
    "x_test":      "/kaggle/input/datasets/thinhdeeptry123/dataset3/X_test.csv",
    "val_pred_file": "/kaggle/working/valid_lgbm.csv",
    "output_file": "/kaggle/working/submission_lgbm_v3.csv",

    # ── Feature config ──────────────────────────────────────────────
    "last_k":      5,
    "use_bigram":  True,
    "n_jobs":      4,          # ← THÊM: parallel workers cho feature extraction

    # ── LightGBM params ─────────────────────────────────────────────
    # ✅ TỐI ƯU TRAINING:
    #   boosting_type : gbdt   → goss    (Gradient One-Side Sampling, ~2-3x faster)
    #   learning_rate : 0.05   → 0.10    (~2x ít rounds hơn)
    #   max_bin       : 255    → 127     (histogram nhỏ hơn → build nhanh hơn ~1.5x)
    #   num_boost_round: 1000  → 500     (lr cao hơn hội tụ nhanh hơn)
    #   early_stopping: 50     → 30
    "lgbm_params": {
        "objective":          "multiclass",
        "boosting_type":      "goss",        # ← ĐỔI: gbdt → goss
        "top_rate":           0.2,           # ← THÊM: giữ 20% large-gradient data
        "other_rate":         0.1,           # ← THÊM: sample 10% small-gradient data
        "learning_rate":      0.10,          # ← ĐỔI: 0.05 → 0.10
        "num_leaves":         127,
        "max_depth":          -1,
        "min_child_samples":  10,
        "feature_fraction":   0.6,
        "reg_alpha":          0.1,
        "reg_lambda":         1.0,
        "max_bin":            127,           # ← THÊM: default 255 → 127
        "is_unbalance":       True,
        "verbose":            -1,
        "seed":               42,
        "n_jobs":             -1,
    },
    "num_boost_round": 500,    # ← ĐỔI: 1000 → 500
    "early_stopping":  30,     # ← ĐỔI: 50 → 30
    "seed":            42,
}

FEATURE_COLS = [f"feature_{i}" for i in range(1, 38)]
TARGET_COLS  = [f"attr_{i}"    for i in range(1, 7)]

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
print("✅ Config sẵn sàng")

✅ Config sẵn sàng


## 2. Load Data

In [7]:
def load_csv(path):
    return pd.read_csv(path, index_col="id")

X_train = load_csv(CFG["x_train"])
Y_train = load_csv(CFG["y_train"])
X_val   = load_csv(CFG["x_val"])
Y_val   = load_csv(CFG["y_val"])
X_test  = load_csv(CFG["x_test"])

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

X_train : (51000, 37)
X_val   : (7200, 37)
X_test  : (38000, 37)


In [8]:
def compute_class_info(Y):
    n_classes_list, imbalance_ratios, label_encoders = [], [], {}
    for col in TARGET_COLS:
        vc   = Y[col].value_counts()
        uniq = sorted(Y[col].unique())
        n_classes_list.append(len(uniq))
        imbalance_ratios.append(vc.max() / vc.min())
        le = LabelEncoder()
        le.fit(uniq)
        label_encoders[col] = le
    return n_classes_list, imbalance_ratios, label_encoders

print("📊 Class Info (Train):")
n_classes_list, imbalance_ratios, label_encoders = compute_class_info(Y_train)
for i, col in enumerate(TARGET_COLS):
    print(f"  {col}: {n_classes_list[i]:>3} classes | IR={imbalance_ratios[i]:>7.1f}")

📊 Class Info (Train):
  attr_1:  12 classes | IR=  111.9
  attr_2:  31 classes | IR=   12.2
  attr_3:  99 classes | IR=    1.2
  attr_4:  12 classes | IR=  110.0
  attr_5:  31 classes | IR=   12.3
  attr_6:  99 classes | IR=    1.2


## 3. Feature Engineering

### 3.1 Build Vocabulary — Optimized

> **Tối ưu**: thay `iterrows()` bằng `X[cols].to_numpy()` + vòng lặp numpy array.
> `iterrows()` tạo 1 `pd.Series` object cho mỗi hàng (rất chậm). Numpy array iteration không có overhead này → **~8x nhanh hơn**.

In [9]:
def build_vocab(X: pd.DataFrame):
    """
    Xây dựng token2idx (unigram) và bigram2idx (bigram).
    ✅ TỐI ƯU: dùng .to_numpy() + pd.isna() thay vì iterrows().
    """
    arr = X[FEATURE_COLS].to_numpy()  # (N, 37) — numpy array, không tạo Series

    token2idx  = {}
    bigram2idx = {}
    uni_idx = bi_idx = 0

    for row in arr:                         # iterate numpy rows — nhanh hơn iterrows ~8x
        tokens = row[~pd.isna(row)]         # filter NaN vectorized
        for t in tokens:
            if t not in token2idx:
                token2idx[t] = uni_idx
                uni_idx += 1
        for a, b in zip(tokens[:-1], tokens[1:]):
            bg = (a, b)
            if bg not in bigram2idx:
                bigram2idx[bg] = bi_idx
                bi_idx += 1

    print(f"  Unigram vocab : {len(token2idx)}")
    print(f"  Bigram  vocab : {len(bigram2idx)}")
    return token2idx, bigram2idx


print("📖 Build Vocabulary từ X_train (KHÔNG dùng X_val)...")
token2idx, bigram2idx = build_vocab(X_train)

📖 Build Vocabulary từ X_train (KHÔNG dùng X_val)...
  Unigram vocab : 254
  Bigram  vocab : 6710


### 3.2 Extract Features — Optimized (Parallel)

> **Tối ưu**:
> 1. `to_numpy()` thay `iterrows()` — tránh tạo Series object mỗi hàng
> 2. Inline append thay vì gọi hàm `add()` — giảm function call overhead
> 3. **`joblib.Parallel`** chia dataset thành chunks, xử lý song song — **~N_JOBS lần nhanh hơn**

In [10]:
def _extract_rows(
    arr_chunk, row_offset,
    token2idx, bigram2idx, last_k, use_bigram,
    V, B, off_bow_full, off_bow_recent, off_bigram, off_lastk, off_agg,
):
    """Xử lý 1 chunk numpy array rows. Trả về (rows, cols, vals) với global index."""
    K = last_k
    rows, cols_list, vals = [], [], []

    for local_i, row in enumerate(arr_chunk):
        tokens  = row[~pd.isna(row)]
        seq_len = len(tokens)
        if seq_len == 0:
            continue

        gi      = row_offset + local_i        # global row index
        encoded = [token2idx.get(t, -1) for t in tokens]
        valid   = [e for e in encoded if e >= 0]

        # 1. BoT full — inline (tránh overhead gọi hàm add())
        for idx in valid:
            rows.append(gi); cols_list.append(off_bow_full + idx); vals.append(1.0)

        # 2. BoT recent (nửa sau)
        half = max(1, seq_len // 2)
        for t in tokens[-half:]:
            idx = token2idx.get(t, -1)
            if idx >= 0:
                rows.append(gi); cols_list.append(off_bow_recent + idx); vals.append(1.0)

        # 3. Bigram BoT
        if use_bigram:
            for a, b in zip(tokens[:-1], tokens[1:]):
                bg_idx = bigram2idx.get((a, b), -1)
                if bg_idx >= 0:
                    rows.append(gi); cols_list.append(off_bigram + bg_idx); vals.append(1.0)

        # 4. Last-K positional
        recent_K = tokens[-K:] if seq_len >= K else tokens
        for pos_from_end, t in enumerate(reversed(recent_K)):
            idx = token2idx.get(t, -1)
            if idx >= 0 and pos_from_end < K:
                rows.append(gi)
                cols_list.append(off_lastk + pos_from_end * V + idx)
                vals.append(1.0)

        # 5. Aggregates
        unique_ratio  = len(set(valid)) / max(seq_len, 1)
        first_norm    = token2idx.get(tokens[0],  0) / max(V, 1)
        last_norm     = token2idx.get(tokens[-1], 0) / max(V, 1)
        cnt = defaultdict(int)
        for e in valid:
            cnt[e] += 1
        mode_tok  = max(cnt, key=cnt.get, default=0) / max(V, 1)
        max_freq  = max(cnt.values(), default=0) / max(seq_len, 1)
        n_unique  = len(cnt) / max(V, 1)
        if cnt:
            probs   = np.array(list(cnt.values()), dtype=float)
            probs  /= probs.sum()
            entropy = -np.sum(probs * np.log(probs + 1e-9))
        else:
            entropy = 0.0
        sec_last = token2idx.get(tokens[-2], 0) / max(V, 1) if seq_len >= 2 else 0.0

        for j, v in enumerate([
            seq_len, unique_ratio, first_norm, last_norm,
            mode_tok, max_freq, n_unique, entropy
        ]):
            rows.append(gi); cols_list.append(off_agg + j); vals.append(float(v))

    return rows, cols_list, vals


def extract_features(
    X: pd.DataFrame,
    token2idx: dict,
    bigram2idx: dict,
    last_k: int = 5,
    use_bigram: bool = True,
    n_jobs: int = 4,
) -> csr_matrix:
    """
    ✅ TỐI ƯU: numpy array iteration + joblib parallel processing.
    prefer='threads' tránh pickle overhead cho sparse matrix merging.
    """
    V = len(token2idx); B = len(bigram2idx); N = len(X)
    K = last_k

    off_bow_full   = 0
    off_bow_recent = V
    off_bigram     = V * 2
    off_lastk      = V * 2 + (B if use_bigram else 0)
    off_agg        = off_lastk + K * V
    total_dim      = off_agg + 8

    arr    = X[FEATURE_COLS].to_numpy()          # convert 1 lần duy nhất
    splits = np.array_split(np.arange(N), min(n_jobs, N))

    chunk_results = Parallel(n_jobs=n_jobs, prefer="threads")(
        delayed(_extract_rows)(
            arr[idx_range], int(idx_range[0]),
            token2idx, bigram2idx, last_k, use_bigram,
            V, B, off_bow_full, off_bow_recent, off_bigram, off_lastk, off_agg,
        )
        for idx_range in splits
    )

    all_rows, all_cols, all_vals = [], [], []
    for r, c, v in chunk_results:
        all_rows.extend(r); all_cols.extend(c); all_vals.extend(v)

    return csr_matrix((all_vals, (all_rows, all_cols)), shape=(N, total_dim))

In [11]:
print("🔧 Trích xuất features (parallel, n_jobs={})...".format(CFG["n_jobs"]))
print("  X_train...", end=" ", flush=True)
X_train_feat = extract_features(X_train, token2idx, bigram2idx,
                                 last_k=CFG["last_k"], use_bigram=CFG["use_bigram"],
                                 n_jobs=CFG["n_jobs"])
print(f"→ {X_train_feat.shape} | nnz={X_train_feat.nnz:,}")

print("  X_val...  ", end=" ", flush=True)
X_val_feat = extract_features(X_val, token2idx, bigram2idx,
                               last_k=CFG["last_k"], use_bigram=CFG["use_bigram"],
                               n_jobs=CFG["n_jobs"])
print(f"→ {X_val_feat.shape}")

print("  X_test... ", end=" ", flush=True)
X_test_feat = extract_features(X_test, token2idx, bigram2idx,
                                last_k=CFG["last_k"], use_bigram=CFG["use_bigram"],
                                n_jobs=CFG["n_jobs"])
print(f"→ {X_test_feat.shape}")

sparsity = 1 - X_train_feat.nnz / (X_train_feat.shape[0] * X_train_feat.shape[1])
print(f"\n  Sparsity: {sparsity:.2%}")

🔧 Trích xuất features (parallel, n_jobs=4)...
  X_train... → (51000, 8496) | nnz=2,097,583
  X_val...   → (7200, 8496)
  X_test...  → (38000, 8496)

  Sparsity: 99.52%


## 4. Label Encoding

In [12]:
Y_train_enc = np.zeros((len(Y_train), 6), dtype=np.int32)
Y_val_enc   = np.zeros((len(Y_val),   6), dtype=np.int32)

for i, col in enumerate(TARGET_COLS):
    le = label_encoders[col]
    Y_train_enc[:, i] = le.transform(Y_train[col])
    Y_val_enc[:, i]   = le.transform(Y_val[col])

print("✅ Label encoding xong")

✅ Label encoding xong


## 5. Train 6 LightGBM Models

### Thay đổi params:
| Param | Bản gốc | Bản tối ưu | Lý do |
|---|---|---|---|
| `boosting_type` | `gbdt` | **`goss`** | Gradient-based One-Side Sampling, ~2-3x nhanh hơn |
| `learning_rate` | `0.05` | **`0.10`** | Hội tụ nhanh hơn ~2x |
| `max_bin` | `255` (default) | **`127`** | Histogram nhỏ hơn → build nhanh ~1.5x |
| `num_boost_round` | `1000` | **`500`** | lr cao → ít rounds |
| `early_stopping` | `50` | **`30`** | Dừng sớm hơn khi plateau |

In [13]:
def get_lgbm_params(col, n_classes, ir, base_params):
    params = dict(base_params)
    params["num_class"] = n_classes

    if n_classes > 50:          # attr_3 & attr_6 (99 classes)
        params["num_leaves"]        = 255
        params["learning_rate"]     = 0.10
        params["is_unbalance"]      = False
        params["min_child_samples"] = 10
        params["feature_fraction"]  = 0.5
    elif ir > 50:               # attr_1 & attr_4 (imbalance nặng)
        params["num_leaves"]        = 127
        params["learning_rate"]     = 0.10
        params["is_unbalance"]      = True
        params["min_child_samples"] = 5
        params["feature_fraction"]  = 0.6
    else:                       # attr_2 & attr_5
        params["num_leaves"]        = 127
        params["learning_rate"]     = 0.10
        params["is_unbalance"]      = True
        params["min_child_samples"] = 10
        params["feature_fraction"]  = 0.6

    return params

print("✅ Hàm get_lgbm_params sẵn sàng")

✅ Hàm get_lgbm_params sẵn sàng


In [14]:
print(f"\n{'='*65}")
print(f"  TRAINING 6 LightGBM Models (GOSS + lr=0.10 + max_bin=127)")
print(f"{'='*65}")
print(f"{'Attr':<8} {'Classes':>8} {'IR':>8} {'BestRound':>10} {'Val Acc':>10}")
print("-" * 50)

models        = []
val_preds_enc = np.zeros_like(Y_val_enc)

for i, col in enumerate(TARGET_COLS):
    n_cls  = n_classes_list[i]
    ir     = imbalance_ratios[i]
    params = get_lgbm_params(col, n_cls, ir, CFG["lgbm_params"])

    dtrain = lgb.Dataset(X_train_feat, label=Y_train_enc[:, i], free_raw_data=False)
    dval   = lgb.Dataset(X_val_feat,   label=Y_val_enc[:, i],
                         reference=dtrain, free_raw_data=False)

    booster = lgb.train(
        params          = params,
        train_set       = dtrain,
        num_boost_round = CFG["num_boost_round"],
        valid_sets      = [dval],
        callbacks       = [
            lgb.early_stopping(CFG["early_stopping"], verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    proba = booster.predict(X_val_feat)
    pred  = np.argmax(proba, axis=1)
    acc   = accuracy_score(Y_val_enc[:, i], pred)
    val_preds_enc[:, i] = pred

    print(f"{col:<8} {n_cls:>8} {ir:>8.1f} "
          f"{booster.best_iteration:>10} {acc:>10.4f}")
    models.append(booster)

em = (val_preds_enc == Y_val_enc).all(axis=1).mean()
print("-" * 50)
print(f"{'Exact-Match Accuracy (Val)':>40}: {em:.4f}")
print(f"{'='*65}")


  TRAINING 6 LightGBM Models (GOSS + lr=0.10 + max_bin=127)
Attr      Classes       IR  BestRound    Val Acc
--------------------------------------------------
attr_1         12    111.9        267     0.9978
attr_2         31     12.2        500     0.9926
attr_3         99      1.2        487     0.9697
attr_4         12    110.0        476     0.9983
attr_5         31     12.3        202     0.9950
attr_6         99      1.2        468     0.9776
--------------------------------------------------
              Exact-Match Accuracy (Val): 0.9508


## 6. Metrics đầy đủ trên Validation Set

> Đo **Exact Match, F1-Score, Precision, Recall** theo 2 cấp độ:
> - **Per-attribute**: metrics riêng từng attr (chi tiết)
> - **Global (gộp 6 attr)**: trung bình weighted across attrs — dùng để so sánh với Transformer / BiGRU như trong bảng paper

In [15]:
val_preds_raw = np.zeros_like(val_preds_enc, dtype=np.int64)
Y_val_raw     = np.zeros_like(Y_val_enc,     dtype=np.int64)

for i, col in enumerate(TARGET_COLS):
    le = label_encoders[col]
    val_preds_raw[:, i] = le.inverse_transform(val_preds_enc[:, i]).astype(np.int64)
    Y_val_raw[:, i]     = le.inverse_transform(Y_val_enc[:, i]).astype(np.int64)

print("✅ Decode labels xong")

✅ Decode labels xong


In [16]:
def compute_metrics(y_true, y_pred, label_name=""):
    """Tính metrics cho 1 attribute."""
    return {
        "label":       label_name,
        "accuracy":    accuracy_score(y_true, y_pred),
        "f1_macro":    f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "f1_micro":    f1_score(y_true, y_pred, average="micro",    zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "prec_macro":  precision_score(y_true, y_pred, average="macro",    zero_division=0),
        "rec_macro":   recall_score(   y_true, y_pred, average="macro",    zero_division=0),
        "prec_wt":     precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "rec_wt":      recall_score(   y_true, y_pred, average="weighted", zero_division=0),
    }


# ── Per-attribute metrics ───────────────────────────────────────────────────
results     = [compute_metrics(Y_val_raw[:, i], val_preds_raw[:, i], col)
               for i, col in enumerate(TARGET_COLS)]
exact_match = (val_preds_raw == Y_val_raw).all(axis=1).mean()

print("=" * 90)
print("  PER-ATTRIBUTE METRICS — LightGBM v3 (Optimized)")
print("=" * 90)
hdr = (f"{'Attribute':<10} {'Accuracy':>9} {'F1-Mac':>8} {'F1-Mic':>8} {'F1-Wtd':>8} "
       f"{'Prec-Mac':>9} {'Rec-Mac':>8} {'Prec-Wtd':>9} {'Rec-Wtd':>8}")
print(hdr)
print("-" * 90)
for m in results:
    print(f"{m['label']:<10} {m['accuracy']:>9.4f} {m['f1_macro']:>8.4f} "
          f"{m['f1_micro']:>8.4f} {m['f1_weighted']:>8.4f} {m['prec_macro']:>9.4f} "
          f"{m['rec_macro']:>8.4f} {m['prec_wt']:>9.4f} {m['rec_wt']:>8.4f}")
print("-" * 90)
print(f"\n  ⭐ EXACT MATCH: {exact_match:.4f}  ({exact_match*100:.2f}%)")
print("=" * 90)


# ══════════════════════════════════════════════════════════════════════════════
# GLOBAL METRICS — gộp cả 6 attr (để so sánh với Transformer / BiGRU)
# ══════════════════════════════════════════════════════════════════════════════
#
# Cách tính: mỗi attr có không gian nhãn khác nhau (attr_1: 12 classes,
# attr_3: 99 classes...) nên KHÔNG flatten trực tiếp (trùng label ID).
# → Tính weighted F1/Prec/Rec cho từng attr → lấy MEAN → 1 con số duy nhất.
# Cách này khớp với cách đánh giá Transformer / BiGRU trong paper.

global_f1_list   = [m["f1_weighted"]  for m in results]
global_prec_list = [m["prec_wt"]      for m in results]
global_rec_list  = [m["rec_wt"]       for m in results]

global_f1   = float(np.mean(global_f1_list))
global_prec = float(np.mean(global_prec_list))
global_rec  = float(np.mean(global_rec_list))

print()
print("=" * 65)
print("  GLOBAL METRICS — Gộp cả 6 attr (so sánh với paper)")
print("=" * 65)
print(f"  {'Metric':<25} {'Value':>10}  Per-attr breakdown")
print("-" * 65)
print(f"  {'Exact Match':<25} {exact_match:>10.4f}")
print(f"  {'F1-Score  (weighted)':<25} {global_f1:>10.4f}  "
      f"[{', '.join(f'{v:.4f}' for v in global_f1_list)}]")
print(f"  {'Precision (weighted)':<25} {global_prec:>10.4f}  "
      f"[{', '.join(f'{v:.4f}' for v in global_prec_list)}]")
print(f"  {'Recall    (weighted)':<25} {global_rec:>10.4f}  "
      f"[{', '.join(f'{v:.4f}' for v in global_rec_list)}]")
print("=" * 65)
print()
print("  📊 So sánh với kết quả paper:")
print(f"  {'Mô hình':<25} {'Exact Match':>12} {'F1-Score':>10} "
      f"{'Precision':>11} {'Recall':>8}")
print(f"  {'─'*66}")
print(f"  {'Transformer':<25} {0.9993:>12.4f} {0.9983:>10.4f} {0.9985:>11.4f} {0.9984:>8.4f}")
print(f"  {'BiGRU':<25} {0.9990:>12.4f} {0.9985:>10.4f} {0.9986:>11.4f} {0.9986:>8.4f}")
print(f"  {'LightGBM (Optimized)':<25} {exact_match:>12.4f} "
      f"{global_f1:>10.4f} {global_prec:>11.4f} {global_rec:>8.4f}  ← KẾT QUẢ NÀY")
print("=" * 65)


  VALIDATION METRICS — LightGBM v3 (Optimized)
Attribute   Accuracy   F1-Mac   F1-Mic   F1-Wtd  Prec-Mac  Rec-Mac  Prec-Wtd  Rec-Wtd
------------------------------------------------------------------------------------------
attr_1        0.9978   0.9655   0.9978   0.9977    0.9712   0.9608    0.9978   0.9978
attr_2        0.9926   0.9889   0.9926   0.9926    0.9889   0.9891    0.9927   0.9926
attr_3        0.9697   0.9693   0.9697   0.9697    0.9695   0.9696    0.9703   0.9697
attr_4        0.9983   0.9703   0.9983   0.9983    0.9732   0.9687    0.9984   0.9983
attr_5        0.9950   0.9923   0.9950   0.9950    0.9923   0.9924    0.9950   0.9950
attr_6        0.9776   0.9775   0.9776   0.9776    0.9776   0.9777    0.9780   0.9776
------------------------------------------------------------------------------------------

  ⭐ EXACT MATCH: 0.9508  (95.08%)


In [17]:
# ── Bảng per-attr (DataFrame) ─────────────────────────────────────────────
df_metrics = pd.DataFrame(results).set_index("label")
df_metrics.index.name = "Attribute"
df_metrics.columns = ["Accuracy", "F1-Macro", "F1-Micro", "F1-Weighted",
                       "Prec-Macro", "Rec-Macro", "Prec-Weighted", "Rec-Weighted"]

# Thêm 2 hàng tổng hợp: MEAN (per-attr avg) và GLOBAL (weighted avg = dùng để so sánh)
df_metrics.loc["MEAN (attr avg)"] = df_metrics.mean()

# GLOBAL row: chỉ cột Weighted mới có ý nghĩa để report
global_row = {
    "Accuracy":      float(np.mean([m["accuracy"]    for m in results])),
    "F1-Macro":      float(np.mean([m["f1_macro"]    for m in results])),
    "F1-Micro":      float(np.mean([m["f1_micro"]    for m in results])),
    "F1-Weighted":   global_f1,
    "Prec-Macro":    float(np.mean([m["prec_macro"]  for m in results])),
    "Rec-Macro":     float(np.mean([m["rec_macro"]   for m in results])),
    "Prec-Weighted": global_prec,
    "Rec-Weighted":  global_rec,
}
df_metrics.loc["GLOBAL (paper)"] = global_row

print("\n📋 Bảng metrics đầy đủ:")
print(df_metrics.round(4).to_string())
print()
print(f"  ⭐ Exact Match      : {exact_match:.4f}")
print(f"  ⭐ F1-Score (global): {global_f1:.4f}   ← con số report trong paper")
print(f"  ⭐ Precision(global): {global_prec:.4f}   ← con số report trong paper")
print(f"  ⭐ Recall   (global): {global_rec:.4f}   ← con số report trong paper")



📋 Bảng metrics (DataFrame):
           Accuracy  F1-Macro  F1-Micro  F1-Weighted  Prec-Macro  Rec-Macro  Prec-Weighted  Rec-Weighted
Attribute                                                                                               
attr_1       0.9978    0.9655    0.9978       0.9977      0.9712     0.9608         0.9978        0.9978
attr_2       0.9926    0.9889    0.9926       0.9926      0.9889     0.9891         0.9927        0.9926
attr_3       0.9697    0.9693    0.9697       0.9697      0.9695     0.9696         0.9703        0.9697
attr_4       0.9983    0.9703    0.9983       0.9983      0.9732     0.9687         0.9984        0.9983
attr_5       0.9950    0.9923    0.9950       0.9950      0.9923     0.9924         0.9950        0.9950
attr_6       0.9776    0.9775    0.9776       0.9776      0.9776     0.9777         0.9780        0.9776
MEAN         0.9885    0.9773    0.9885       0.9885      0.9788     0.9764         0.9887        0.9885

  Exact-Match Accuracy: 0

In [18]:
for i, col in enumerate(TARGET_COLS):
    print(f"\n{'─'*60}")
    print(f"📊 Classification Report — {col}")
    print(f"{'─'*60}")
    print(classification_report(Y_val_raw[:, i], val_preds_raw[:, i],
                                 zero_division=0, digits=4))


────────────────────────────────────────────────────────────
📊 Classification Report — attr_1
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

           1     1.0000    1.0000    1.0000      1038
           2     0.9600    0.8571    0.9057        28
           3     0.9992    1.0000    0.9996      1287
           4     0.9429    0.8684    0.9041        38
           5     1.0000    1.0000    1.0000       659
           6     0.9062    0.8788    0.8923        33
           7     0.9024    0.9250    0.9136        40
           8     1.0000    1.0000    1.0000       512
           9     1.0000    1.0000    1.0000      1494
          10     0.9459    1.0000    0.9722        35
          11     1.0000    1.0000    1.0000       762
          12     0.9977    1.0000    0.9988      1274

    accuracy                         0.9978      7200
   macro avg     0.9712    0.9608    0.9655      7200
weighted avg     0.9978    0.997

## 7. Lưu file dự đoán Validation

In [19]:
df_val_pred = pd.DataFrame(val_preds_raw, columns=TARGET_COLS, index=X_val.index)
df_val_pred.index.name = "id"
df_val_pred = df_val_pred.astype(np.uint16)
df_val_pred.to_csv(CFG["val_pred_file"])
print(f"✅ Đã lưu dự đoán val: {CFG['val_pred_file']}")
print(f"   Shape: {df_val_pred.shape}")
print("\nPreview (5 dòng đầu):")
print(df_val_pred.head())

df_loaded = pd.read_csv(CFG["val_pred_file"], index_col="id")
em_check  = (df_loaded.values == Y_val_raw).all(axis=1).mean()
print(f"\nSanity check — Exact Match từ file: {em_check:.4f}  ✅")

✅ Đã lưu dự đoán val: /kaggle/working/valid_lgbm.csv
   Shape: (7200, 6)

Preview (5 dòng đầu):
       attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
id                                                   
gsmrs       5       1      59      12       1      42
a6j96      12      19       5      12      31      15
ix1nt      12      23      56       5       6      35
phyp3      12      12      64       3      14      68
zyzm7      11       1      78      12       1      17

Sanity check — Exact Match từ file: 0.9508  ✅


## 8. Predict Test Set & Tạo Submission

In [20]:
def make_submission(models, X_test_feat, label_encoders, X_test_index, output_path):
    preds_enc = np.zeros((X_test_feat.shape[0], 6), dtype=np.int32)
    for i, booster in enumerate(models):
        proba = booster.predict(X_test_feat)
        preds_enc[:, i] = np.argmax(proba, axis=1)

    preds_raw = np.zeros_like(preds_enc, dtype=np.uint16)
    for i, col in enumerate(TARGET_COLS):
        preds_raw[:, i] = label_encoders[col].inverse_transform(
            preds_enc[:, i]
        ).astype(np.uint16)

    sub = pd.DataFrame(preds_raw, columns=TARGET_COLS, index=X_test_index)
    sub.index.name = "id"
    sub = sub.astype(np.uint16)
    sub.to_csv(output_path)
    print(f"✅ Submission saved: {output_path}")
    print(sub.head())
    return sub


print("🔮 Predict X_test...")
sub = make_submission(models, X_test_feat, label_encoders,
                      X_test.index, CFG["output_file"])

🔮 Predict X_test...
✅ Submission saved: /kaggle/working/submission_lgbm_v3.csv
       attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
id                                                   
gpbfd      11       1      84      12       1      96
w22ee       9      19      55       3      12      63
wyw95       1       1      98       3       1      53
izx4w       5       1      72       1       1      77
c6o2d      11       1      22      12       1       1


## 9. Tóm tắt kết quả

In [21]:
print("=" * 65)
print("  TÓM TẮT KẾT QUẢ CUỐI CÙNG")
print("=" * 65)
print()
print("  [PER-ATTRIBUTE]")
for m in results:
    print(f"    {m['label']} → Acc={m['accuracy']:.4f}  "
          f"F1-wt={m['f1_weighted']:.4f}  "
          f"Prec-wt={m['prec_wt']:.4f}  "
          f"Rec-wt={m['rec_wt']:.4f}")
print()
print("  [GLOBAL — dùng để so sánh với Transformer / BiGRU]")
print(f"    Exact Match : {exact_match:.4f}  ({exact_match*100:.2f}%)")
print(f"    F1-Score    : {global_f1:.4f}")
print(f"    Precision   : {global_prec:.4f}")
print(f"    Recall      : {global_rec:.4f}")
print()
print("  📊 Bảng so sánh với paper:")
print(f"  {'Mô hình':<25} {'Exact Match':>12} {'F1-Score':>10} "
      f"{'Precision':>11} {'Recall':>8}")
print(f"  {'─'*66}")
print(f"  {'Transformer':<25} {0.9993:>12.4f} {0.9983:>10.4f} {0.9985:>11.4f} {0.9984:>8.4f}")
print(f"  {'BiGRU':<25} {0.9990:>12.4f} {0.9985:>10.4f} {0.9986:>11.4f} {0.9986:>8.4f}")
print(f"  {'LightGBM (Optimized)':<25} {exact_match:>12.4f} "
      f"{global_f1:>10.4f} {global_prec:>11.4f} {global_rec:>8.4f}")
print(f"  {'─'*66}")
print()
print(f"  📁 Val predictions : {CFG['val_pred_file']}")
print(f"  📁 Test submission : {CFG['output_file']}")
print("=" * 65)


  TÓM TẮT KẾT QUẢ
  Exact Match Accuracy (Val) : 0.9508  (95.08%)

  attr_1 → Acc=0.9978  F1-wt=0.9977  Prec-wt=0.9978  Rec-wt=0.9978
  attr_2 → Acc=0.9926  F1-wt=0.9926  Prec-wt=0.9927  Rec-wt=0.9926
  attr_3 → Acc=0.9697  F1-wt=0.9697  Prec-wt=0.9703  Rec-wt=0.9697
  attr_4 → Acc=0.9983  F1-wt=0.9983  Prec-wt=0.9984  Rec-wt=0.9983
  attr_5 → Acc=0.9950  F1-wt=0.9950  Prec-wt=0.9950  Rec-wt=0.9950
  attr_6 → Acc=0.9776  F1-wt=0.9776  Prec-wt=0.9780  Rec-wt=0.9776

  📁 Val predictions : /kaggle/working/valid_lgbm.csv
  📁 Test submission : /kaggle/working/submission_lgbm_v3.csv
